In [1]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/stock_price
!ls



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/stock_price
stock-price-prediction-feature-preprocess
stock-price-prediction-feature-preprocess.zip


In [ ]:
!unzip stock-price-prediction-feature-preprocess.zip -d stock-price-prediction-feature-preprocess

In [2]:
%cd /content/drive/MyDrive/stock_price/stock-price-prediction-feature-preprocess
!ls

/content/drive/MyDrive/stock_price/stock-price-prediction-feature-preprocess
arima_output  stock-price-prediction-feature-preprocess


In [3]:
%cd stock-price-prediction-feature-preprocess
!ls

/content/drive/MyDrive/stock_price/stock-price-prediction-feature-preprocess/stock-price-prediction-feature-preprocess
arima_output  data  README.md  requirements.txt  stock_preprocess.ipynb


In [4]:
!pip install -r requirements.txt

In [11]:
import os
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tools.sm_exceptions import ConvergenceWarning

warnings.simplefilter("ignore", ConvergenceWarning)

In [20]:
project_path = "/content/drive/MyDrive/stock_price/stock-price-prediction-feature-preprocess/stock-price-prediction-feature-preprocess"

files = [
    project_path+"/data/FPT_preprocessed.csv",
    project_path+"/data/HPG_preprocessed.csv",
    project_path+"/data/VCB_preprocessed.csv",
    project_path+"/data/VIC_preprocessed.csv",
    project_path+"/data/VNM_preprocessed.csv",
]

output_dir = os.path.join(project_path, "arima_output")
os.makedirs(output_dir, exist_ok=True)

print("Output folder:", output_dir)

Output folder: /content/drive/MyDrive/stock_price/stock-price-prediction-feature-preprocess/stock-price-prediction-feature-preprocess/arima_output


In [21]:
for file in files:
    print(file, "->", os.path.exists(file))

/content/drive/MyDrive/stock_price/stock-price-prediction-feature-preprocess/stock-price-prediction-feature-preprocess/data/FPT_preprocessed.csv -> True
/content/drive/MyDrive/stock_price/stock-price-prediction-feature-preprocess/stock-price-prediction-feature-preprocess/data/HPG_preprocessed.csv -> True
/content/drive/MyDrive/stock_price/stock-price-prediction-feature-preprocess/stock-price-prediction-feature-preprocess/data/VCB_preprocessed.csv -> True
/content/drive/MyDrive/stock_price/stock-price-prediction-feature-preprocess/stock-price-prediction-feature-preprocess/data/VIC_preprocessed.csv -> True
/content/drive/MyDrive/stock_price/stock-price-prediction-feature-preprocess/stock-price-prediction-feature-preprocess/data/VNM_preprocessed.csv -> True


In [22]:
def determine_d(series, max_d=2):
    current_series = series.copy().dropna()

    for d in range(max_d + 1):
        try:
            result = adfuller(current_series)
            p_value = result[1]

            if p_value <= 0.05:
                return d

            current_series = current_series.diff().dropna()

        except:
            return d

    return max_d

In [23]:
def find_best_arima_order(series, d, max_p=3, max_q=3):
    best_aic = np.inf
    best_order = None

    for p in range(max_p + 1):
        for q in range(max_q + 1):
            try:
                model = ARIMA(
                    series,
                    order=(p, d, q),
                    enforce_stationarity=False,
                    enforce_invertibility=False
                )

                model_fit = model.fit()

                converged = model_fit.mle_retvals.get("converged", True)

                if not converged:
                    continue

                if model_fit.aic < best_aic:
                    best_aic = model_fit.aic
                    best_order = (p, d, q)

            except Exception:
                continue

    if best_order is None:
        best_order = (1, d, 1)

    return best_order

In [25]:
for file in files:
    df = pd.read_csv(file)

    code = df["stock_code"].iloc[0]

    df["trading_date"] = pd.to_datetime(df["trading_date"])
    df = df.sort_values("trading_date")

    for typee in ["close", "log_close"]:
        for rangee in [1, 5, 20]:

            data = df[["trading_date", typee]].copy()

            data.columns = ["ds", "y"]
            data["ds"] = pd.to_datetime(data["ds"])
            data = data.dropna()

            series = data.set_index("ds")["y"]

            # Gán frequency ngày làm việc để giảm warning
            series = series.asfreq("B")
            series = series.ffill()

            d = determine_d(series)

            best_order = find_best_arima_order(
                series,
                d=d,
                max_p=3,
                max_q=3
            )

            model = ARIMA(
                series,
                order=best_order,
                enforce_stationarity=False,
                enforce_invertibility=False
            )

            model = ARIMA(
    series,
    order=best_order,
    enforce_stationarity=False,
    enforce_invertibility=False
)

            model_fit = model.fit()

            # Predict cho toàn bộ phần quá khứ
            in_sample_result = model_fit.get_prediction(
                start=0,
                end=len(series) - 1
            )

            in_sample_mean = in_sample_result.predicted_mean
            in_sample_conf = in_sample_result.conf_int()

            history = pd.DataFrame({
                "ds": series.index,
                "y": series.values,
                "yhat": in_sample_mean.values,
                "yhat_lower": in_sample_conf.iloc[:, 0].values,
                "yhat_upper": in_sample_conf.iloc[:, 1].values
            })

            # Forecast cho tương lai
            forecast_result = model_fit.get_forecast(steps=rangee)

            forecast_mean = forecast_result.predicted_mean
            conf_int = forecast_result.conf_int()

            future_dates = pd.bdate_range(
                start=series.index[-1] + pd.offsets.BDay(1),
                periods=rangee
            )

            future = pd.DataFrame({
                "ds": future_dates,
                "y": [np.nan] * rangee,
                "yhat": forecast_mean.values,
                "yhat_lower": conf_int.iloc[:, 0].values,
                "yhat_upper": conf_int.iloc[:, 1].values
            })

            check = pd.concat([history, future], ignore_index=True)

            stock_output_dir = os.path.join(output_dir, code)
            os.makedirs(stock_output_dir, exist_ok=True)

            path = os.path.join(stock_output_dir, f"arima_{rangee}d_{code}_{typee}.csv")
            check.to_csv(path, index=False)

            print(f"ARIMA {best_order} - {code} - {typee} - {rangee}d")
            print(check.tail(20))
            print("Saved:", path)
            print("-" * 60)

ARIMA (2, 1, 3) - FPT - close - 1d
             ds      y        yhat  yhat_lower  yhat_upper
3588 2025-11-07  99.96   99.104980   97.585488  100.624472
3589 2025-11-10  95.21  100.071706   98.552214  101.591198
3590 2025-11-11  95.11   94.963038   93.443546   96.482530
3591 2025-11-12  99.56   95.189247   93.669755   96.708740
3592 2025-11-13  97.88   99.927139   98.407647  101.446631
3593 2025-11-14  98.97   97.484799   95.965307   99.004291
3594 2025-11-17  99.96   98.957709   97.438217  100.477201
3595 2025-11-18  98.97  100.345145   98.825653  101.864637
3596 2025-11-19  96.99   98.713945   97.194453  100.233437
3597 2025-11-20  97.98   96.711288   95.191796   98.230781
3598 2025-11-21  99.76   98.421402   96.901909   99.940894
3599 2025-11-24  99.17   99.826641   98.307149  101.346133
3600 2025-11-25  98.67   98.705526   97.186034  100.225018
3601 2025-11-26  97.98   98.860865   97.341372  100.380357
3602 2025-11-27  98.48   98.249331   96.729839   99.768823
3603 2025-11-28  96.1

In [26]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import pandas as pd
import os

eval_results = []

for file in os.listdir(output_dir):
    if file.endswith(".csv") and file.startswith("arima_"):
        file_path = os.path.join(output_dir, file)

        df_eval = pd.read_csv(file_path)
        df_eval = df_eval.dropna(subset=["y", "yhat"])

        if len(df_eval) == 0:
            continue

        if "log_close" in file:
            y_true = np.exp(df_eval["y"])
            y_pred = np.exp(df_eval["yhat"])
            target_eval = "close_from_log_close"
        else:
            y_true = df_eval["y"]
            y_pred = df_eval["yhat"]
            target_eval = "close"

        mae = mean_absolute_error(y_true, y_pred)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

        eval_results.append({
            "file": file,
            "target_eval": target_eval,
            "MAE": mae,
            "RMSE": rmse,
            "MAPE": mape
        })

eval_df = pd.DataFrame(eval_results)

eval_path = os.path.join(output_dir, "arima_eval.csv")
eval_df.to_csv(eval_path, index=False)

eval_df

""
